# Neural Networks & Architectures
**Level:** Advanced | **Interactive Machine Learning Curriculum**

> Perceptrons, Multi-Layer Perceptrons (MLPs), Activation Functions (Sigmoid, Tanh, ReLU, GELU, Swish), and the Universal Approximation Theorem.

---

### 📖 Conceptual Foundations

#### What is an Artificial Neural Network?

An **Artificial Neural Network (ANN)** is a computational model composed of interconnected layers of artificial neurons (nodes). Each connection carries a scalar **weight** ($W$), and each neuron computes a weighted sum of inputs plus a **bias** ($b$), passing the result through a non-linear **activation function** $\sigma(z)$:

$$z = \sum_{i=1}^{n} w_i x_i + b = \mathbf{w}^T \mathbf{x} + b$$
$$a = \sigma(z)$$

Stacking these operations across multiple layers enables the network to approximate virtually any continuous function.


#### The Perceptron and the XOR Crisis

In 1957, Frank Rosenblatt introduced the **Perceptron**, a binary classifier with a step activation function:

$$\hat{y} = \begin{cases} 1 & \text{if } \mathbf{w}^T \mathbf{x} + b \ge 0 \\ 0 & \text{otherwise} \end{cases}$$

In 1969, Marvin Minsky and Seymour Papert published their famous critique proving that a single-layer perceptron could not learn non-linearly separable functions like the **XOR (Exclusive OR)** logic gate. This triggered the first 'AI Winter' until researchers formalized that adding hidden layers with non-linear activations overcomes this limitation completely.


#### Universal Approximation Theorem (Cybenko, 1989; Hornik, 1991)

The **Universal Approximation Theorem** states that a feedforward network with a single hidden layer containing a finite number of neurons and a non-linear activation function (like sigmoid or ReLU) can approximate any continuous function on compact subsets of $\mathbb{R}^n$ to any arbitrary degree of precision $\epsilon > 0$.

> **Key Caveat:** The theorem is an *existence proof*, not an *algorithmic guarantee*. It does not tell us how many neurons are needed (it could require exponentially many) or whether gradient descent can find the optimal weights.


#### Comprehensive Guide to Activation Functions

| Activation Function | Mathematical Formula | Output Range | Derivative Range | Key Advantages & Common Drawbacks |
| --- | --- | --- | --- | --- |
| Sigmoid (Logistic) | $\sigma(z) = \frac{1}{1 + e^{-z}}$ | $(0, 1)$ | $(0, 0.25]$ | Smooth probability interpretation; suffers from severe vanishing gradients for $|z| > 4$; not zero-centered. |
| Hyperbolic Tangent (Tanh) | $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$ | $(-1, 1)$ | $(0, 1.0]$ | Zero-centered gradients improve convergence over Sigmoid; still suffers from vanishing gradients at saturation tails. |
| ReLU (Rectified Linear Unit) | $f(z) = \max(0, z)$ | $[0, \infty)$ | $\{0, 1\}$ | Cheap to compute, constant gradient 1 for positive inputs preventing vanishing gradients; vulnerable to 'Dying ReLU' problem. |
| Leaky ReLU | $f(z) = \max(\alpha z, z), \; \alpha \approx 0.01$ | $(-\infty, \infty)$ | $\{\alpha, 1\}$ | Prevents dying neurons by providing a small gradient $\alpha$ for negative inputs. |
| ELU (Exponential Linear) | $f(z) = z \text{ if } z > 0 \text{ else } \alpha(e^z - 1)$ | $(-\alpha, \infty)$ | Smooth everywhere | Brings mean activations closer to zero; computationally more expensive due to exponential. |
| GELU (Gaussian Error Linear) | $f(z) = z \cdot \Phi(z) \approx 0.5z(1 + \tanh(\sqrt{2/\pi}(z + 0.044715z^3)))$ | $(-0.17, \infty)$ | Smooth, non-monotonic | Standard in modern Transformers (BERT, GPT, ViT); probabilistic gating based on input magnitude. |
| Swish / SiLU | $f(z) = z \cdot \sigma(\beta z)$ | $(-0.28, \infty)$ | Smooth, self-gated | Discovered via automated search by Google Brain; outperforms ReLU in deep networks (EfficientNet). |


#### The 'Dying ReLU' Pathology

If a neuron's weights are updated such that $\mathbf{w}^T \mathbf{x} + b < 0$ for **all** training examples, the neuron outputs 0 and its gradient is identically 0:

$$\frac{\partial \text{Loss}}{\partial w_i} = \frac{\partial \text{Loss}}{\partial a} \cdot 0 \cdot x_i = 0$$

Because the gradient is zero, gradient descent will never update its weights again. The neuron is effectively permanently dead. Mitigations: lower the learning rate, use He initialization, or adopt Leaky ReLU / GELU.


#### Matrix Formulation of Forward Propagation

For layer $l$ with weight matrix $W^{[l]} \in \mathbb{R}^{n^{[l]} \times n^{[l-1]}}$, bias vector $\mathbf{b}^{[l]} \in \mathbb{R}^{n^{[l]}}$, and batch input $A^{[l-1]} \in \mathbb{R}^{n^{[l-1]} \times m}$ (where $m$ is batch size):

$$Z^{[l]} = W^{[l]} A^{[l-1]} + \mathbf{b}^{[l]}$$
$$A^{[l]} = g^{[l]}(Z^{[l]})$$

Vectorized batching executes via highly optimized Basic Linear Algebra Subprograms (BLAS) or Tensor Cores on GPUs, achieving trillions of floating point operations per second (TFLOPS).



In [ ]:
# %matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Activation functions
def relu(Z): return np.maximum(0, Z)
def sigmoid(Z): return 1.0 / (1.0 + np.exp(-np.clip(Z, -25, 25)))

# Architecture: 2 inputs -> 4 hidden neurons -> 1 output neuron
n_x, n_h, n_y = 2, 4, 1
m = 200 # sample points

# He / Xavier initialization
W1 = np.random.randn(n_h, n_x) * np.sqrt(2.0 / n_x)
b1 = np.zeros((n_h, 1))
W2 = np.random.randn(n_y, n_h) * np.sqrt(1.0 / n_h)
b2 = np.zeros((n_y, 1))

# Generate XOR-like non-linear dataset
X = np.random.randn(n_x, m)
Y = ((X[0, :] * X[1, :] > 0).astype(int)).reshape(1, m)

# Forward propagation
Z1 = np.dot(W1, X) + b1
A1 = relu(Z1)
Z2 = np.dot(W2, A1) + b2
A2 = sigmoid(Z2)

# Binary Cross Entropy Loss
loss = -np.mean(Y * np.log(A2 + 1e-12) + (1 - Y) * np.log(1 - A2 + 1e-12))
print(f'Initial Forward Pass Loss: {loss:.4f}')
print(f'A1 shape: {A1.shape}, A2 shape: {A2.shape}')

# Plot activations distribution
plt.figure(figsize=(7, 4))
plt.hist(A1.flatten(), bins=20, color='skyblue', edgecolor='black', alpha=0.7)
plt.title('Distribution of Hidden Layer Activations (ReLU)')
plt.xlabel('Activation Value')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

